In [98]:
!pip install google-adk --quiet

In [99]:
import os
import logging
import requests
from typing import Optional, List, Dict

import vertexai
from google.adk.agents import Agent, LlmAgent
from google.adk.tools import agent_tool
from google.adk.tools.google_search_tool import google_search
from google.adk.runners import InMemoryRunner
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-00-11b9d8ae6979"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [100]:
from google.cloud import secretmanager

def get_secret(secret_id: str) -> str:
    """Retrieve a secret value from Google Secret Manager.

    Args:
        secret_id (str): The name of the secret to retrieve.

    Returns:
        str: The decoded secret string value.
    """
    sm_client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT_ID}/secrets/{secret_id}/versions/latest"
    response = sm_client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")

GOOGLE_MAPS_API_KEY = get_secret("GOOGLE_MAPS_API_KEY")
print("API key loaded from Secret Manager")

API key loaded from Secret Manager


In [101]:
def get_lat_lon(city: str, state: str) -> Optional[Dict[str, float]]:
    """Use the Google Maps Geocoding API to convert a city and state to latitude and longitude.

    Args:
        city (str): The name of the city (e.g., "Austin").
        state (str): The two-letter state abbreviation (e.g., "TX").

    Returns:
        Optional[Dict[str, float]]: A dictionary with 'lat' and 'lon' keys,
        or None if the location could not be geocoded.
    """
    address = f"{city}, {state}"
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": address, "key": GOOGLE_MAPS_API_KEY}

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Geocoding API error: {response.status_code}")
        return None

    data = response.json()
    if data.get("status") != "OK" or not data.get("results"):
        print(f"No geocoding results for: {address}")
        return None

    location = data["results"][0]["geometry"]["location"]
    return {"lat": location["lat"], "lon": location["lng"]}


def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Fetch the extended weather forecast from the U.S. National Weather Service (NWS) API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast dictionaries for each time period,
        each containing:
            - 'name': Name of the forecast period (e.g., "Today", "Tonight")
            - 'startTime': ISO timestamp for the start of the forecast period
            - 'temperature': Temperature value
            - 'temperatureUnit': Temperature unit (e.g., "F" or "C")
            - 'windSpeed': Wind speed description
            - 'windDirection': Wind direction (e.g., "NW")
            - 'shortForecast': Short summary (e.g., "Partly Sunny")
            - 'detailedForecast': Full text forecast description
        Returns None if the forecast could not be retrieved.
    """
    headers = {
        "User-Agent": "MyWeatherApp (student-00-eab28a1c4614@qwiklabs.net)",
        "Accept": "application/geo+json"
    }

    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    response = requests.get(points_url, headers=headers)
    if response.status_code != 200:
        print(f"Error fetching data from points endpoint: {response.status_code}")
        return None

    points_data = response.json()
    forecast_url = points_data["properties"].get("forecast")
    if not forecast_url:
        print("Forecast URL not found in response.")
        return None

    forecast_response = requests.get(forecast_url, headers=headers)
    if forecast_response.status_code != 200:
        print(f"Error fetching forecast: {forecast_response.status_code}")
        return None

    forecast_data = forecast_response.json()
    periods = forecast_data["properties"].get("periods", [])

    return [
        {
            "name": p.get("name", ""),
            "startTime": p.get("startTime", ""),
            "temperature": str(p.get("temperature", "")),
            "temperatureUnit": p.get("temperatureUnit", ""),
            "windSpeed": p.get("windSpeed", ""),
            "windDirection": p.get("windDirection", ""),
            "shortForecast": p.get("shortForecast", ""),
            "detailedForecast": p.get("detailedForecast", ""),
        }
        for p in periods
    ]

In [102]:
WEATHER_AGENT_INSTRUCTIONS = """
You are a specialized weather assistant. Your only job is to provide
weather forecasts and conditions for US cities.

When a user asks about the weather in a city:
1. Use the get_lat_lon tool to convert the city and state to latitude and longitude.
2. Use the get_extended_weather_forecast tool with those coordinates to get the forecast.
3. Summarize the current conditions clearly, including temperature, wind, and outlook.
4. If conditions are severe or dangerous, clearly flag this as a WEATHER ALERT.

IMPORTANT: If the user asks about ANYTHING other than weather (news, facts, general
questions, etc.), you must transfer back to the root_agent immediately. Do not attempt
to answer questions outside of weather.

Be concise, friendly, and helpful. Use plain language, not raw data dumps.
If a tool fails or returns no data, let the user know politely.
"""

In [103]:
weather_agent = LlmAgent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description=(
        "A specialized agent that retrieves and summarizes real-time US weather "
        "forecasts using the National Weather Service API. Use this agent for any "
        "weather-related questions."
    ),
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

In [104]:
SEARCH_AGENT_INSTRUCTIONS = """
You are a research assistant with access to Google Search.
Your job is to find accurate, up-to-date information to answer the user's question.

When given a question:
1. Use the google_search tool to search for current information.
2. Summarize the search results clearly and concisely.
3. Always base your answers on the search results — never rely on training data
   alone for current events, sports scores, or facts that may have changed.
4. Cite key details from what you found.

Do not answer from memory. Always search first.
"""

In [105]:
google_search_agent = LlmAgent(
    name="google_search_agent",
    model="gemini-2.5-flash",
    description=(
        "A research agent that uses Google Search to find current information, "
        "news, sports scores, facts, and general knowledge. Use this agent for "
        "any non-weather question that requires up-to-date information."
    ),
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
)

In [107]:
root_agent = Agent(
    name="root_agent",
    model="gemini-2.5-flash",
    description="A helpful assistant that delegates weather questions to the weather agent and all other questions to the search agent.",
    tools=[
        agent_tool.AgentTool(agent=weather_agent),
        agent_tool.AgentTool(agent=google_search_agent),
    ],
)

In [108]:
async def run_multi_agent(user_input: str) -> str:
    """Send a message to the multi-agent system and return the final response.
    Prints detailed events showing which sub-agents are being called.

    Args:
        user_input (str): The user's message.

    Returns:
        str: The final agent response text.
    """
    runner = InMemoryRunner(agent=root_agent, app_name="multi_agent_app")
    session = await runner.session_service.create_session(
        app_name="multi_agent_app", user_id="user_01"
    )
    message = types.Content(role="user", parts=[types.Part(text=user_input)])

    response_text = ""
    print(f"\n--- Events for: '{user_input}' ---")

    for event in runner.run(
        user_id="user_01",
        session_id=session.id,
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                # Detect when root agent calls a sub-agent tool
                if hasattr(part, 'function_call') and part.function_call:
                    print(f"  [DELEGATION] {event.author} → calling tool: {part.function_call.name}")
                    if part.function_call.args:
                        # Show truncated input to the tool
                        args_str = str(dict(part.function_call.args))
                        print(f"              input: {args_str[:150]}...")

                # Detect when a sub-agent tool returns its result
                if hasattr(part, 'function_response') and part.function_response:
                    print(f"  [RESPONSE]   ← result from: {part.function_response.name}")

        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text

    return response_text

In [109]:
from IPython.display import Markdown, display

test_queries = [
    ("🌤 WEATHER", "What's the weather like in Nashville, TN today?"),
    ("🔍 SEARCH",  "Who won the most recent Super Bowl?"),
    ("🌤 WEATHER", "Will I need an umbrella in Portland, OR this weekend?"),
    ("🔍 SEARCH",  "What are the latest developments in quantum computing?"),
    ("🌤 WEATHER", "Any severe weather alerts for Oklahoma City, OK?"),
]

print("=" * 60)
print("CHALLENGE 3 — Multi-Agent System Test")
print("=" * 60)

for expected, query in test_queries:
    print(f"\n{expected} | You: {query}")
    response = await run_multi_agent(query)
    display(Markdown(f"**Agent:** {response}"))
    print("-" * 60)

CHALLENGE 3 — Multi-Agent System Test

🌤 WEATHER | You: What's the weather like in Nashville, TN today?

--- Events for: 'What's the weather like in Nashville, TN today?' ---
  [DELEGATION] root_agent → calling tool: weather_agent
              input: {'request': "What's the weather like in Nashville, TN today?"}...
  [RESPONSE]   ← result from: weather_agent


**Agent:** Today in Nashville, TN, expect a high of 78°F. There's a chance of showers and thunderstorms, mainly before 3 PM, with mostly cloudy skies. Winds will be from the south-southwest at 10 to 15 mph, with gusts up to 25 mph.

**WEATHER ALERT**: There is a chance of showers and thunderstorms this afternoon.

------------------------------------------------------------

🔍 SEARCH | You: Who won the most recent Super Bowl?

--- Events for: 'Who won the most recent Super Bowl?' ---
  [DELEGATION] root_agent → calling tool: google_search_agent
              input: {'request': 'Who won the most recent Super Bowl?'}...
  [RESPONSE]   ← result from: google_search_agent


**Agent:** The Seattle Seahawks won the most recent Super Bowl, Super Bowl LX, held on February 8, 2026. They defeated the New England Patriots with a score of 29-13 in Santa Clara, marking their second Super Bowl championship.

------------------------------------------------------------

🌤 WEATHER | You: Will I need an umbrella in Portland, OR this weekend?

--- Events for: 'Will I need an umbrella in Portland, OR this weekend?' ---
  [DELEGATION] root_agent → calling tool: weather_agent
              input: {'request': 'Will I need an umbrella in Portland, OR this weekend?'}...
  [RESPONSE]   ← result from: weather_agent


...

------------------------------------------------------------

🔍 SEARCH | You: What are the latest developments in quantum computing?

--- Events for: 'What are the latest developments in quantum computing?' ---
  [DELEGATION] root_agent → calling tool: google_search_agent
              input: {'request': 'latest developments in quantum computing'}...
  [RESPONSE]   ← result from: google_search_agent


**Agent:** Quantum computing is seeing rapid advancements with significant breakthroughs, a clearer understanding of its applications, and ongoing efforts to overcome challenges. The field is shifting from theoretical exploration to practical, real-world solutions.

**Key Breakthroughs and Developments:**
*   **Google's Quantum AI** reportedly demonstrated a quantum computer outperforming supercomputers by 13,000 times in specific calculations in late 2025 and showcased below-threshold quantum error correction in February 2026.
*   **IonQ** achieved a record 99.99% two-qubit gate fidelity in 2025.
*   The emergence of **room-temperature quantum computing** technologies from companies like IonQ and Xanadu, which could lower costs and increase accessibility.
*   **Fujitsu and RIKEN** announced a 256-qubit superconducting quantum computer and plan for a 1,000-qubit system by late 2026.
*   **QuantWare** is developing a 3D wiring architecture for 10,000-qubit processors by 2028.
*   **Microsoft's** Majorana 1 Chip is the first quantum processor to use topological qubits for enhanced scalability and reduced errors.
*   A collaboration between **Fermilab and MIT Lincoln Laboratory** achieved a breakthrough in controlling ion traps using in-vacuum cryoelectronics.
*   **Quantum Computing Inc. (QCi)** expanded its portfolio by acquiring NuCrypt and Luminar Semiconductor in early 2026.

**Ongoing Challenges:**
*   **Qubit Stability and Error Correction:** Qubits are fragile, and while error correction is progressing, fault-tolerant systems are complex and require many physical qubits per logical qubit.
*   **Scaling Qubit Numbers:** Current systems have 50-200 qubits, but tens of thousands to millions are needed for practical applications. IBM aims for 2,000 logical qubits by 2033, and Google targets a million by 2030.
*   **Hardware and Infrastructure:** The high cost and complexity of specialized cooling systems for many quantum computers remain a hurdle.
*   **Software and Algorithm Development:** Quantum software tools are still nascent, making algorithm development and debugging difficult.
*   **Skills Gap:** There's a significant shortage of skilled professionals in the field.
*   **Cost and Accessibility:** High costs raise concerns about equitable access.

**Emerging Applications:**
*   **Drug Discovery and Materials Science:** Simulating molecular behavior to accelerate discovery of new drugs and materials.
*   **Cybersecurity:** Quantum Key Distribution (QKD) offers secure communication, but quantum computing also poses a threat to current encryption, driving the need for post-quantum cryptography.
*   **Financial Modeling and Optimization:** Optimizing portfolios, risk assessment, fraud detection, and market simulations.
*   **Artificial Intelligence and Machine Learning:** Boosting AI capabilities, accelerating ML algorithms, and developing advanced generative quantum AI.
*   **Logistics and Supply Chain Management:** Optimizing complex systems.
*   Other areas include climate modeling, weather forecasting, traffic optimization, and quantum sensing.

Major players include IBM, Google, Microsoft, Amazon, IonQ, Rigetti Computing, and others, alongside national laboratories and universities. The outlook suggests that quantum computing is maturing rapidly, with a focus on transitioning from laboratory breakthroughs to practical applications within the next decade, with the market projected to reach $1.3 trillion by 2035.

------------------------------------------------------------

🌤 WEATHER | You: Any severe weather alerts for Oklahoma City, OK?

--- Events for: 'Any severe weather alerts for Oklahoma City, OK?' ---
  [DELEGATION] root_agent → calling tool: weather_agent
              input: {'request': 'Any severe weather alerts for Oklahoma City, OK?'}...
  [RESPONSE]   ← result from: weather_agent


**Agent:** There are no severe weather alerts for Oklahoma City, OK.

For this afternoon, expect patchy fog before 3 PM, then a chance of showers and thunderstorms. The high will be around 74°F with a south wind of 10 to 14 mph, gusting up to 21 mph. There's a 30% chance of precipitation.

Tonight, there's a chance of showers and thunderstorms before 9 PM, becoming likely between 9 PM and midnight, then a chance of showers and thunderstorms again. The low will be around 63°F with a south wind of 13 to 17 mph, gusting up to 26 mph. There's a 70% chance of precipitation with new rainfall between a quarter and half of an inch possible.

------------------------------------------------------------
